# Divye FE: Frequency + OOF Target Encoding + Interactions

Inspired by Divye's public notebook. Three layers of new numeric features:

1. **Frequency encoding** — for every feature, how common is this value? (train+test, no target leakage)
2. **OOF target encoding** — per-fold mean target rate per value, computed on training fold only (prevents leakage)
3. **Multiplicative interactions** — pairwise products of the top-8 TE features by |correlation| with target

These are fundamentally different from the manual interaction features tried before (`s6e2-feature-engineering.ipynb`):  
- The previous attempt used string-concatenation interactions — CatBoost already learns these natively.  
- TE features are **smooth numeric probability signals** — CatBoost sees them differently from raw categoricals.

**Baseline**: catboost_default_cpu  OOF AUC = 0.95544

In [5]:
import subprocess
import numpy as np
import pandas as pd
from itertools import combinations
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(train)
test  = prep(test)

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES

X      = train[ALL_FEATURES]
y      = train['heart_disease'].values
X_test = test[ALL_FEATURES]

SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f'Train: {X.shape}  Test: {X_test.shape}')

Train: (630000, 13)  Test: (270000, 13)


## Feature Engineering Functions

In [6]:
def compute_freq_features(train_df, test_df, cols):
    """Frequency of each value in the combined train+test pool.
    No target info — safe to compute on full data."""
    combined = pd.concat([train_df[cols], test_df[cols]])
    tr_freq, te_freq = {}, {}
    for col in cols:
        freq_map = combined[col].value_counts(normalize=True)
        tr_freq[f'{col}_freq'] = train_df[col].map(freq_map).fillna(0).values
        te_freq[f'{col}_freq'] = test_df[col].map(freq_map).fillna(0).values
    return tr_freq, te_freq


def compute_oof_te(train_df, test_df, cols, y, skf):
    """OOF target encoding. Keys are '{col}_te' to avoid overwriting original columns.
    - Train: each row uses TE from its held-out fold (no leakage).
    - Test: average of per-fold TE maps (reduces fold-specific noise).
    """
    global_mean = y.mean()
    oof_te  = {f'{col}_te': np.zeros(len(train_df)) for col in cols}
    test_te = {f'{col}_te': np.zeros(len(test_df))  for col in cols}

    for col in cols:
        key       = f'{col}_te'
        tr_vals   = train_df[col].values
        te_vals   = test_df[col].values
        fold_test = []

        for tr_idx, val_idx in skf.split(train_df, y):
            te_map = (pd.DataFrame({'v': tr_vals[tr_idx], 't': y[tr_idx]})
                        .groupby('v')['t'].mean())
            oof_te[key][val_idx] = (pd.Series(tr_vals[val_idx])
                                      .map(te_map).fillna(global_mean).values)
            fold_test.append(pd.Series(te_vals).map(te_map).fillna(global_mean).values)

        test_te[key] = np.mean(fold_test, axis=0)

    return oof_te, test_te


def select_top_interactions(oof_te, y, n=8):
    """Return the top-n TE feature names by |Pearson correlation| with target."""
    corrs = {col: abs(np.corrcoef(vals, y)[0, 1]) for col, vals in oof_te.items()}
    top = sorted(corrs, key=corrs.get, reverse=True)[:n]
    print('Top TE features by |corr| with target:')
    for col in top:
        print(f'  {col:40s}  corr={corrs[col]:.4f}')
    return top


def make_interaction_features(te_dict, top_cols):
    """Pairwise multiplicative interactions of top TE features."""
    out = {}
    for c1, c2 in combinations(top_cols, 2):
        name = f'{c1}_x_{c2}'
        out[name] = te_dict[c1] * te_dict[c2]
    return out


# --- Precompute OOF TE (to select top-8 interaction features) ---
print('Computing frequency encoding...')
tr_freq, te_freq = compute_freq_features(X, X_test, ALL_FEATURES)

print('Computing OOF target encoding (5 folds × 13 features)...')
oof_te, test_te = compute_oof_te(X, X_test, ALL_FEATURES, y, SKF)

print('\nSelecting top-8 TE features for interactions...')
top8 = select_top_interactions(oof_te, y, n=8)

tr_inter = make_interaction_features(oof_te,  top8)
te_inter = make_interaction_features(test_te, top8)
print(f'\n{len(tr_inter)} interaction features: {list(tr_inter.keys())[:3]} ...')

Computing frequency encoding...
Computing OOF target encoding (5 folds × 13 features)...

Selecting top-8 TE features for interactions...
Top TE features by |corr| with target:
  thallium_te                               corr=0.6058
  chest_pain_type_te                        corr=0.5252
  max_hr_te                                 corr=0.4784
  number_of_vessels_fluro_te                corr=0.4632
  st_depression_te                          corr=0.4489
  exercise_angina_te                        corr=0.4419
  slope_of_st_te                            corr=0.4298
  sex_te                                    corr=0.3424

28 interaction features: ['thallium_te_x_chest_pain_type_te', 'thallium_te_x_max_hr_te', 'thallium_te_x_number_of_vessels_fluro_te'] ...


## Build Augmented Feature Matrix

In [7]:
def build_augmented(base_df, freq_dict, te_dict, inter_dict):
    """Combine original + freq + TE + interaction features into a DataFrame."""
    df = base_df.copy().reset_index(drop=True)
    for name, vals in {**freq_dict, **te_dict, **inter_dict}.items():
        df[name] = vals
    return df


X_aug      = build_augmented(X,      tr_freq, oof_te,  tr_inter)
X_test_aug = build_augmented(X_test, te_freq, test_te, te_inter)

print(f'Augmented train: {X_aug.shape}  test: {X_test_aug.shape}')
print(f'New features: {X_aug.shape[1] - X.shape[1]}')
print(f'  Freq:         {len(tr_freq)}')
print(f'  TE:           {len(oof_te)}')
print(f'  Interactions: {len(tr_inter)}')

Augmented train: (630000, 67)  test: (270000, 67)
New features: 54
  Freq:         13
  TE:           13
  Interactions: 28


## 5-Fold CV with Per-Fold TE Recomputation

The augmented X_aug above uses global OOF TE (correct for no leakage on OOF rows).  
For the CV we recompute TE within each fold to ensure val rows only see training-fold TE.

In [8]:
CATBOOST_PARAMS = dict(
    iterations        = 2000,
    learning_rate     = 0.05,
    depth             = 6,
    task_type         = 'CPU',
    thread_count      = -1,
    cat_features      = CAT_FEATURES,
    random_seed       = 42,
    verbose           = 0,
)

global_mean = y.mean()
oof_preds   = np.zeros(len(y))
test_folds  = np.zeros((5, len(X_test)))

for fold, (tr_idx, val_idx) in enumerate(SKF.split(X, y)):
    print(f'\n=== Fold {fold + 1}/5 ===')

    X_tr_base  = X.iloc[tr_idx].reset_index(drop=True)
    X_val_base = X.iloc[val_idx].reset_index(drop=True)
    y_tr, y_val = y[tr_idx], y[val_idx]

    # Freq encoding (no target info — same logic regardless of fold)
    fold_tr_freq, fold_te_freq  = compute_freq_features(X_tr_base, X_test,      ALL_FEATURES)
    _,            fold_val_freq = compute_freq_features(X_tr_base, X_val_base,   ALL_FEATURES)

    # Per-fold TE: build map from training fold only, apply to val and test
    # Keys use '_te' suffix to avoid overwriting original categorical columns
    fold_tr_te, fold_val_te, fold_te_te = {}, {}, {}
    for col in ALL_FEATURES:
        key    = f'{col}_te'
        te_map = (pd.DataFrame({'v': X_tr_base[col].values, 't': y_tr})
                    .groupby('v')['t'].mean())
        fold_tr_te[key]  = X_tr_base[col].map(te_map).fillna(global_mean).values
        fold_val_te[key] = X_val_base[col].map(te_map).fillna(global_mean).values
        fold_te_te[key]  = X_test[col].map(te_map).fillna(global_mean).values

    # Interactions using per-fold TE (top8 keys are already '{col}_te')
    fold_tr_inter  = make_interaction_features(fold_tr_te,  top8)
    fold_val_inter = make_interaction_features(fold_val_te, top8)
    fold_te_inter  = make_interaction_features(fold_te_te,  top8)

    X_tr_aug  = build_augmented(X_tr_base,  fold_tr_freq,  fold_tr_te,  fold_tr_inter)
    X_val_aug = build_augmented(X_val_base, fold_val_freq, fold_val_te, fold_val_inter)
    X_te_aug  = build_augmented(X_test,     fold_te_freq,  fold_te_te,  fold_te_inter)

    m = CatBoostClassifier(**CATBOOST_PARAMS)
    m.fit(X_tr_aug, y_tr,
          eval_set=(X_val_aug, y_val),
          early_stopping_rounds=50)

    oof_preds[val_idx] = m.predict_proba(X_val_aug)[:, 1]
    test_folds[fold]   = m.predict_proba(X_te_aug)[:, 1]

    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f'Fold {fold + 1}  AUC={fold_auc:.5f}  best_iter={m.best_iteration_}')

cv_auc     = roc_auc_score(y, oof_preds)
test_preds = test_folds.mean(axis=0)

print(f'\nOOF AUC (divye_fe): {cv_auc:.5f}')
print(f'Baseline (cpu):     0.95544')
print(f'Delta:              {cv_auc - 0.95544:+.5f}')


=== Fold 1/5 ===
Fold 1  AUC=0.95605  best_iter=468

=== Fold 2/5 ===
Fold 2  AUC=0.95490  best_iter=498

=== Fold 3/5 ===
Fold 3  AUC=0.95578  best_iter=481

=== Fold 4/5 ===
Fold 4  AUC=0.95538  best_iter=559

=== Fold 5/5 ===
Fold 5  AUC=0.95620  best_iter=599

OOF AUC (divye_fe): 0.95566
Baseline (cpu):     0.95544
Delta:              +0.00022


## Save Submission

In [11]:
import numpy as np

# Save OOF for stacking
np.save('submissions/oof_divye_fe.npy', oof_preds)
print(f'OOF saved: submissions/oof_divye_fe.npy  (AUC={roc_auc_score(y, oof_preds):.5f})')

label = 'catboost_divye_fe'
fname = f'submissions/{label}.csv'

sub = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)
print(f'Saved: {fname}')

desc = f'{label} | cv_auc={cv_auc:.4f}'
r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'{label}: {status}')
print(f'desc: {desc}')

OOF saved: submissions/oof_divye_fe.npy  (AUC=0.95566)
Saved: submissions/catboost_divye_fe.csv
catboost_divye_fe: submitted
desc: catboost_divye_fe | cv_auc=0.9557
